# AnyTime Universal Benchmark

Run early-exit benchmark for any AnyTime model. Switch by changing **one import line** below.

All HW metrics **per-PID** (process-scoped). Energy = `Î£ (power_w_i Ã— end_to_end_sec_i)` in Joules.

Per-run dir contains:
- `hw_results.json` â€” latency + memory + energy + per-PID GPU/CPU + FLOPs/params
- `quality_results.json` â€” accuracy / F1 / mAP / PPL (only if `skip_quality=False`)

Default sweep = HW-only. Pass `skip_quality=False` to enable quality eval.

Outputs:
- `logs/benchmark/{model}/...` â€” per-run JSONs (nested 2-3 levels deep)
- `results/{model}/` â€” CSV summaries


## 1. Setup

In [1]:
import numpy as np

In [2]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

repo root: D:\earlyexit\EarlyExitMonoRepo


In [3]:
# HF login (only needed if pulling from a private repo)
from shared import load_env

load_env()

try:
    from google.colab import userdata

    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN", ""))
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF: logged in")
else:
    print("HF: no token, public repos only")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF: logged in


## 2. Choose model

**Change ONE import line below** to swap which model gets benchmarked.

In [4]:
# # === SWAP THIS LINE TO PICK MODEL ===
# # from benchmark_config import bert as cfg
# from benchmark_config import llama  as cfg
# # from benchmark_config import vision as cfg
# # from benchmark_config import yolo   as cfg

# print("model :", cfg.NAME)
# print("family:", cfg.MODEL_FAMILY)
# print("out   :", cfg.OUT_DIR)

## 3. Run full sweep

Default = HW-only (`skip_quality=True`). Each run does:
- `profile_hw()` â€” HW + latency + per-PID power/VRAM/CPU + FLOPs/params + EDP
- `evaluate_quality()` â€” (skipped by default)

Override sweep dims via `only_*` kwargs (see `cfg.run_all` signature).


In [5]:
# cfg.run_all(skip_quality=False)          # HW + quality (all datasets)
# # cfg.run_all()                            # HW-only (cnn_dailymail latency only)
# # cfg.run_all(skip_quality=False, only_exit=4)  # single exit smoke test

## 3b. Run ALL models (full sweep)

In [6]:
# # Run ALL models -- HW + quality sweep for every model family.
# # Comment out any model you want to skip.
# from benchmark_config import bert, llama, vision, yolo

# for _cfg in [bert, llama, vision, yolo]:
#     print(f"{60*chr(61)}")
#     print(f"  {_cfg.NAME} ({_cfg.MODEL_FAMILY})")
#     print(60*chr(61))
#     try:
#         _cfg.run_all(skip_quality=False,dry_run=True)  # HW + quality (all datasets)
#     except Exception as _e:
#         print(f"[{_cfg.NAME}] run_all failed: {_e}")
#         import traceback; traceback.print_exc()


In [7]:
import re
from shared import write_benchmark_csvs, write_average_csvs
from benchmark_config import bert, llama, vision, yolo
from collections import defaultdict

# Export CSVs for ALL models from local logs/benchmark/{model}/...
# Recursive walk, merges hw_results.json + quality_results.json per run dir,
# groups by dataset/mode parent, exit_0,1,2,... ordered (not lexical), then
# writes per-model cross-task averages (mean+std) under results/{model}/average/.

def _exit_key(m):
    nums = [int(n) for n in re.findall(r'\d+', m)]
    return nums or [10**9]

for _cfg in [bert, llama, vision, yolo]:
    out_dir = Path(_cfg.OUT_DIR)
    csv_root = REPO_ROOT / 'results' / _cfg.NAME
    csv_root.mkdir(parents=True, exist_ok=True)

    hw_dirs = {p.parent for p in out_dir.rglob('hw_results.json')}
    q_dirs  = {p.parent for p in out_dir.rglob('quality_results.json')}
    run_dirs = list(hw_dirs | q_dirs)
    if not run_dirs:
        print(f'[{_cfg.NAME}] no results found, skipping')
        continue
    print(f'[{_cfg.NAME}] {len(run_dirs)} run dirs ({len(hw_dirs)} hw, {len(q_dirs)} quality)')

    groups = defaultdict(dict)
    for rd in run_dirs:
        rel_parent = rd.parent.relative_to(out_dir).as_posix()
        group_key = rel_parent.replace('/', '_') or 'root'
        groups[group_key][rd.name] = rd

    for group_key, runs in sorted(groups.items()):
        csv_dir = csv_root / group_key
        csv_dir.mkdir(parents=True, exist_ok=True)
        ordered = sorted(runs.keys(), key=_exit_key)
        write_benchmark_csvs(
            results_files=runs,
            out_dir=csv_dir,
            baseline_key=None,
            method_order=ordered,
        )
        print(f'  {group_key}: {len(runs)} runs -> {csv_dir}')

    # Per-model cross-task averages (mean+std over tasks) -> results/{model}/average/
    write_average_csvs(csv_root)

print('All CSVs under:', REPO_ROOT / 'results')


[bert] no results found, skipping
[llama] no results found, skipping
[vision] no results found, skipping
[yolo] no results found, skipping
All CSVs under: D:\earlyexit\EarlyExitMonoRepo\results


## 4b. Export CSVs — ALL models

## 4. Export CSVs

Recursive walk over per-run dirs (handles 2-3 level nesting per model). Merges
`hw_results.json` + `quality_results.json` if both exist. Groups by parent dir.


## 5. Quick view

In [8]:
import pandas as pd

results_root = REPO_ROOT / 'results'
for csv_name in ('latency.csv', 'energy.csv', 'quality.csv', 'hardware.csv'):
    for f in results_root.rglob(csv_name):
        rel = f.relative_to(results_root)
        print(f'=== {rel} ===')
        try:
            print(pd.read_csv(f).head(8))
        except Exception as e:
            print(f'(read failed: {e})')
        print()


In [9]:
from shared import plot_model_panel
from pathlib import Path

# One PNG per metric (+ quality / perplexity-log / higher-better) per model,
# averaged across tasks with mean +/- std bands. YOLO sub-exits (P3/P4/P5)
# split into separate coloured lines. Saved to results/{model}/plots/.
results_root = REPO_ROOT / 'results'
for model_dir in sorted(p for p in results_root.iterdir() if p.is_dir()):
    print(f'=== plotting {model_dir.name} ===')
    try:
        plot_model_panel(model_dir)
    except Exception as e:
        print(f'[{model_dir.name}] plot failed: {e}')

print('plots written under', results_root)


=== plotting bert ===
[plot] no CSVs under D:\earlyexit\EarlyExitMonoRepo\results\bert
=== plotting llama ===
[plot] no CSVs under D:\earlyexit\EarlyExitMonoRepo\results\llama
=== plotting vision ===
[plot] no CSVs under D:\earlyexit\EarlyExitMonoRepo\results\vision
=== plotting yolo ===
[plot] no CSVs under D:\earlyexit\EarlyExitMonoRepo\results\yolo
plots written under D:\earlyexit\EarlyExitMonoRepo\results


## 6. Cross-device comparison (A100 vs Jetson)

Reads `logs/` (A100) + `logs.jetson/` (Jetson), pairs runs by
(backend, task, weight_source), and writes `results_compare/`:
- **compare**: task present on BOTH devices -> overlay plot + CSV
- **A100 only / Jetson only**: single-device "normal" output (no skip)
- `average_by_model.xlsx`: per-model metrics averaged across all its tasks
  (one sheet per backend) for copy-paste.

`force=False` skips a group whose output dir already exists (Jetson sweep is
partial -> a re-run only does newly-arrived tasks). Set `force=True` to redo all.

In [10]:
# ensure xlsx engine present (so the average exports as .xlsx, not csv fallback)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"], check=False)

# import as a module so edits to compare_devices.py reload without a kernel restart
import importlib, compare_devices
importlib.reload(compare_devices)

# exports results/ (A100) + results.jetson/ via the SAME exporter benchmark.ipynb
# uses, then double-plots both averaged sets -> results_compare/{model}.png + xlsx
compare_devices.run(force=False)   # force=True to re-export csvs

[export] D:\earlyexit\EarlyExitMonoRepo\logs\benchmark missing; skip
[export] D:\earlyexit\EarlyExitMonoRepo\logs.jetson\benchmark missing; skip
[xlsx] wrote D:\earlyexit\EarlyExitMonoRepo\results_compare\average_by_model.xlsx
[done] plots + xlsx under D:\earlyexit\EarlyExitMonoRepo\results_compare


In [11]:
# show the double plots (A100 vs Jetson, averaged across all tasks) + xlsx sheets
import pandas as pd
from IPython.display import Image, display

cmp_root = REPO_ROOT / 'results_compare'
xlsx = cmp_root / 'average_by_model.xlsx'
if xlsx.exists():
    for name, df in pd.read_excel(xlsx, sheet_name=None).items():
        print(f'=== {name} (A100 vs Jetson, avg across tasks) ==='); display(df)

for png in sorted(cmp_root.glob('*.png')):
    print(png.name); display(Image(str(png)))

=== empty (A100 vs Jetson, avg across tasks) ===


,note
0,no data
